In [ ]:
#@title 1 · Clone repo, install deps, upload data
import subprocess, os, sys

# ── Clone the hybrid branch ─────────────────────────────────────────
!git clone -b cursor/corrected-model-momentum-speed https://github.com/magilliam27/MCI-GRU.git /content/MCI-GRU
os.chdir("/content/MCI-GRU")

# ── Install dependencies ────────────────────────────────────────────
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-geometric
!pip install -q hydra-core omegaconf scipy scikit-learn pandas numpy matplotlib tqdm fredapi python-dateutil

# ── Upload the 2019 universe CSV (~68 MB) ────────────────────────────
from google.colab import files
os.makedirs("data/raw/market", exist_ok=True)
print("sp500_2019_universe_data_through_2026.csv when prompted:")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, "data/raw/market/sp500_2019_universe_data_through_2026.csv")
print("✓ Data file in place")

Cloning into '/content/MCI-GRU'...
remote: Enumerating objects: 625, done.
remote: Counting objects: 100% (625/625), done.
remote: Compressing objects: 100% (362/362), done.
remote: Total 625 (delta 297), reused 541 (delta 218), pack-reused 0 (from 0)
Receiving objects: 100% (625/625), 14.78 MiB | 34.63 MiB/s, done.
Resolving deltas: 100% (297/297), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 4.9 MB/s eta 0:00:00
sp500_2019_universe_data_through_2026.csv when prompted:


Saving sp500_2019_universe_data_through_2026.csv to sp500_2019_universe_data_through_2026.csv
✓ Data file in place


In [ ]:
#@title 1b · Build regime inputs (LSEG copper/oil + FRED yields/market)
!pip install -q fredapi
import os
import numpy as np
import pandas as pd

# ── FRED API key: set here, via env var, or in Colab Secrets ────────
FRED_API_KEY = ""  # paste your key here if not using Secrets
if not FRED_API_KEY:
    FRED_API_KEY = os.environ.get("FRED_API_KEY", "")
if not FRED_API_KEY:
    try:
        from google.colab import userdata
        FRED_API_KEY = userdata.get("FRED_API_KEY")
    except Exception:
        pass
if not FRED_API_KEY:
    raise ValueError(
        "FRED API key required. Set FRED_API_KEY in this cell, as an env var, "
        "or add it to Colab Secrets (key name: FRED_API_KEY). "
        "Get a free key at https://fred.stlouisfed.org/docs/api/api_key.html"
    )
os.environ["FRED_API_KEY"] = FRED_API_KEY

from fredapi import Fred
fred = Fred(api_key=FRED_API_KEY)

# ── Load LSEG partial export (daily copper + oil) ──────────────────
LSEG_PATH = "data/raw/market/regime_inputs_partial_2016_2025.csv"
lseg = pd.read_csv(LSEG_PATH)
lseg["dt"] = pd.to_datetime(lseg["dt"])
print(f"LSEG partial: {len(lseg)} rows, cols={list(lseg.columns)}")

# ── Fetch FRED series (market, yields) with buffer before 2016 ─────
start, end = "2015-12-01", "2025-12-31"
print("Fetching FRED series...")
sp500 = fred.get_series("SP500", observation_start=start, observation_end=end)
dgs10 = fred.get_series("DGS10", observation_start=start, observation_end=end)
dgs3m = fred.get_series("DGS3MO", observation_start=start, observation_end=end)

base = pd.DataFrame({
    "regime_market": sp500,
    "yield_10y": dgs10,
    "yield_3m": dgs3m,
}).sort_index().replace(".", pd.NA).apply(pd.to_numeric, errors="coerce")
base = base.ffill().bfill()
base = base.reset_index().rename(columns={base.reset_index().columns[0]: "dt"})
base["dt"] = pd.to_datetime(base["dt"])

# ── Merge LSEG copper + oil onto FRED dates ───────────────────────
lseg_merge = lseg[["dt", "regime_copper", "regime_oil"]].drop_duplicates(subset=["dt"], keep="last")
base = base.merge(lseg_merge, on="dt", how="outer").sort_values("dt").drop_duplicates(subset=["dt"], keep="last")

for col in ["regime_market", "yield_10y", "yield_3m", "regime_copper", "regime_oil"]:
    base[col] = pd.to_numeric(base[col], errors="coerce").ffill().bfill()

# ── Derived columns ───────────────────────────────────────────────
base["regime_yield_curve"] = base["yield_10y"] - base["yield_3m"]
market_ret = base["regime_market"].pct_change()
yield_chg  = base["yield_10y"].diff()
base["regime_stock_bond_corr"] = market_ret.rolling(63, min_periods=21).corr(yield_chg)

# ── 1-day lag: value at date T uses data from T-1 ─────────────────
regime_cols = [
    "regime_market", "regime_yield_curve", "regime_oil",
    "regime_copper", "regime_stock_bond_corr",
]
base[regime_cols] = base[regime_cols].shift(1)
base = base.dropna(subset=regime_cols, how="all")

regime_df = base[["dt"] + regime_cols].copy()
regime_df["dt"] = regime_df["dt"].dt.strftime("%Y-%m-%d")

out_path = "data/raw/market/regime_inputs_reconciled.csv"
regime_df.to_csv(out_path, index=False)

print(f"✓ Saved {out_path} ({len(regime_df)} rows)")
print(f"  Date range: {regime_df['dt'].min()} → {regime_df['dt'].max()}")
print(f"  Non-null counts:")
for c in regime_cols:
    print(f"    {c}: {regime_df[c].notna().sum()}")

In [ ]:
cmd = f"python run_experiment.py {BASE} experiment_name=seedtest_s{seed} seed={seed}"
proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("RETURNCODE:", proc.returncode)
print("\nSTDOUT:\n", proc.stdout[-8000:])
print("\nSTDERR:\n", proc.stderr[-8000:])

RETURNCODE: 1

STDOUT:
 

STDERR:
 mismatched input '=' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.



In [ ]:
#@title 2 · Run ablation: hybrid vs paper-faithful vs defaults
import subprocess, time

# Shared overrides: use 2019 universe data, trim epochs/models for speed
SHARED = (
    "data.source=csv "
    "data.filename=data/raw/market/sp500_2019_universe_data.csv "
    "data.train_start=2019-01-01 data.train_end=2023-12-31 "
    "data.val_start=2024-01-01   data.val_end=2024-12-31 "
    "data.test_start=2025-01-01  data.test_end=2025-12-31 "
    "training.num_models=3 training.num_epochs=15 "
    "training.early_stopping_patience=5 "
    "seed=54"
)

configs = {
    # ── Recommended hybrid: no multi-scale, self-attn ON, ELU, rank labels
    "hybrid": (
        "experiment_name=hybrid "
        "model.use_multi_scale=false model.use_self_attention=true "
        "model.activation=elu model.latent_init_scale=0.02 "
        "training.label_type=rank "
    ),
    # ── Paper-faithful: no multi-scale, no self-attn, ReLU, raw returns
    "paper_faithful": (
        "experiment_name=paper_faithful "
        "model.use_multi_scale=false model.use_self_attention=false "
        "model.activation=relu model.latent_init_scale=0.02 "
        "training.label_type=returns "
    ),
    # ── Current defaults: multi-scale ON, self-attn ON, ELU, raw returns
    "defaults_with_fixes": (
        "experiment_name=defaults_with_fixes "
        "model.use_multi_scale=true model.use_self_attention=true "
        "model.activation=elu model.latent_init_scale=0.02 "
        "training.label_type=returns "
    ),
}

results = {}
for name, overrides in configs.items():
    print(f"\n{'='*80}")
    print(f"  RUNNING: {name}")
    print(f"{'='*80}\n")
    t0 = time.time()
    cmd = f"python run_experiment.py {SHARED} {overrides}"
    proc = subprocess.run(cmd, shell=True, capture_output=False)
    elapsed = time.time() - t0
    results[name] = {"returncode": proc.returncode, "elapsed_min": elapsed / 60}
    print(f"\n  {name} finished in {elapsed/60:.1f} min  (exit {proc.returncode})")

print(f"\n{'='*80}")
print("  SUMMARY")
print(f"{'='*80}")
for name, r in results.items():
    status = "OK" if r["returncode"] == 0 else "FAIL"
    print(f"  {name:25s}  {status}  ({r['elapsed_min']:.1f} min)")


  RUNNING: hybrid


  hybrid finished in 12.2 min  (exit 0)

  RUNNING: paper_faithful


  paper_faithful finished in 12.0 min  (exit 0)

  RUNNING: defaults_with_fixes


  defaults_with_fixes finished in 12.8 min  (exit 0)

  SUMMARY
  hybrid                     OK  (12.2 min)
  paper_faithful             OK  (12.0 min)
  defaults_with_fixes        OK  (12.8 min)


In [ ]:
#@title 3 · Compare val/test metrics across runs
import glob, json, os
import pandas as pd

rows = []
for exp_dir in sorted(glob.glob("results/*/2*")):
    exp_name = exp_dir.split(os.sep)[-2]
    log_files = glob.glob(os.path.join(exp_dir, "training_*.log"))
    if not log_files:
        continue
    log = open(log_files[0]).read()
    # Extract best val losses from log
    val_losses = []
    for line in log.splitlines():
        if "Best validation loss" in line or "best_val_loss" in line.lower():
            try:
                val_losses.append(float(line.split(":")[-1].strip().split(",")[0].strip(" []")))
            except:
                pass
    # Look for IC / metrics in log
    mean_val = sum(val_losses) / len(val_losses) if val_losses else None
    rows.append({"config": exp_name, "mean_val_loss": mean_val,
                 "num_models": len(val_losses), "log": log_files[0]})

df = pd.DataFrame(rows)
if len(df) > 0:
    print(df.to_string(index=False))
else:
    print("No results found yet -- check results/ directory")

# Print last 30 lines of each log for detailed metrics
for _, row in df.iterrows():
    print(f"\n{'='*60}")
    print(f"  {row['config']}  ({row['log']})")
    print(f"{'='*60}")
    lines = open(row["log"]).readlines()
    print("".join(lines[-30:]))

             config  mean_val_loss  num_models                                                                      log
defaults_with_fixes       0.001302           1 results/defaults_with_fixes/20260224_212423/training_20260224_212423.log
             hybrid       0.083400           1              results/hybrid/20260224_210021/training_20260224_210021.log
     paper_faithful       0.001291           1      results/paper_faithful/20260224_211223/training_20260224_211223.log

  defaults_with_fixes  (results/defaults_with_fixes/20260224_212423/training_20260224_212423.log)
  gradient_clip: 1.0
  loss_type: mse
  ic_loss_alpha: 0.5
  label_type: returns
experiment_name: defaults_with_fixes
output_dir: results
seed: 54

2026-02-24 21:24:23,208 - INFO - 
Random seed: 54
2026-02-24 21:24:23,209 - INFO - Output directory: /content/MCI-GRU/results/defaults_with_fixes/20260224_212423
2026-02-24 21:24:23,212 - INFO - Configuration saved to: /content/MCI-GRU/results/defaults_with_fixes/20260224_

In [ ]:
#@title 4 · Backtest all three configs
import glob, os, subprocess

experiments = ["hybrid", "paper_faithful", "defaults_with_fixes"]

for exp in experiments:
    exp_root = f"results/{exp}"
    if not os.path.isdir(exp_root):
        print(f"⚠ {exp_root} not found, skipping")
        continue

    runs = sorted(glob.glob(os.path.join(exp_root, "2*")))
    if not runs:
        print(f"⚠ No run directories in {exp_root}, skipping")
        continue
    latest = runs[-1]
    pred_dir = os.path.join(latest, "averaged_predictions")

    if not os.path.isdir(pred_dir):
        print(f"⚠ {pred_dir} not found, skipping")
        continue

    n_files = len(glob.glob(os.path.join(pred_dir, "*.csv")))
    print(f"\n{'='*80}")
    print(f"  BACKTEST: {exp}  ({n_files} prediction days)")
    print(f"  {pred_dir}")
    print(f"{'='*80}\n")

    cmd = (
        f"python tests/backtest_sp500.py "
        f"--predictions_dir {pred_dir} "
        f"--data_file data/raw/market/sp500_2019_universe_data.csv "
        f"--test_start 2025-01-01 "
        f"--test_end 2025-12-31 "
        f"--top_k 10 "
        f"--auto_save "
        f"--plot "
        f"--label_t 5"
    )
    subprocess.run(cmd, shell=True)

print(f"\n{'='*80}")
print("  ALL BACKTESTS COMPLETE")
print(f"{'='*80}")


  BACKTEST: hybrid  (250 prediction days)
  results/hybrid/20260224_210021/averaged_predictions


  BACKTEST: paper_faithful  (250 prediction days)
  results/paper_faithful/20260224_211223/averaged_predictions


  BACKTEST: defaults_with_fixes  (250 prediction days)
  results/defaults_with_fixes/20260224_212423/averaged_predictions


  ALL BACKTESTS COMPLETE


In [ ]:
#@title Seed Test: Base model + regime features across seeds
import subprocess, time, os, glob, ast
import numpy as np

# ── Base config: momentum + global regime features ────────────────
BASE = (
"data.source=csv "
"data.filename=data/raw/market/sp500_2019_universe_data_through_2026.csv "
"data.train_start=2019-01-01 data.train_end=2023-12-31 "
"data.val_start=2024-01-01 data.val_end=2024-12-31 "
"data.test_start=2025-01-01 data.test_end=2025-12-31 "
"features=with_momentum "
"features.include_weekly_momentum=false "
"features.momentum_encoding=binary "
"features.momentum_blend_mode=dynamic "
"features.momentum_dynamic_correction_fast_weight=0.15 "
"features.momentum_dynamic_rebound_fast_weight=0.7 "
"features.momentum_dynamic_lookback_periods=0 "
"features.momentum_dynamic_min_history=252 "
"features.momentum_dynamic_min_state_observations=3 "
"features.include_global_regime=true "
"features.regime_inputs_csv='data/raw/market/regime_inputs_reconciled.csv' "
"features.regime_enforce_lag_days=0 "
"model.his_t=60 model.label_t=21 "
"model.use_multi_scale=false model.use_self_attention=false "
"model.activation=elu model.latent_init_scale=0.02 "
"training.label_type=rank "
"training.num_models=10 training.num_epochs=100 "
"training.early_stopping_patience=15 "
)

# Mix of "good" and historically "bad" seeds
SEEDS = [73]

results = {}
for seed in SEEDS:
    print(f"\n{'='*80}")
    print(f"  SEED {seed}")
    print(f"{'='*80}\n")
    t0 = time.time()
    cmd = f"python run_experiment.py {BASE} experiment_name=seedtest_s{seed} seed={seed}"
    proc = subprocess.run(cmd, shell=True, capture_output=False)
    elapsed = time.time() - t0

    # Extract val losses from log
    exp_root = f"results/seedtest_s{seed}"
    val_losses = []
    if os.path.isdir(exp_root):
        runs = sorted(glob.glob(os.path.join(exp_root, "2*")))
        if runs:
            log_files = glob.glob(os.path.join(runs[-1], "training_*.log"))
            if log_files:
                for line in open(log_files[0]):
                    if "Best validation losses" in line:
                        try:
                            raw = "[" + line.split("[")[-1].split("]")[0] + "]"
                            val_losses = list(ast.literal_eval(raw))
                        except:
                            pass

    mean_vl = sum(val_losses) / len(val_losses) if val_losses else None
    results[seed] = {"val_losses": val_losses, "mean": mean_vl, "elapsed": elapsed}
    status = f"mean_val_loss={mean_vl:.6f}" if mean_vl else "PARSE ERROR"
    print(f"\n  Seed {seed}: {status}  ({elapsed/60:.1f} min)")

# ── Summary ────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  SEED TEST SUMMARY (base features, no enhancements)")
print(f"{'='*80}")
print(f"  {'Seed':>6}  {'Mean Val Loss':>14}  {'Min':>12}  {'Max':>12}  {'Spread%':>8}")
print(f"  {'-'*6}  {'-'*14}  {'-'*12}  {'-'*12}  {'-'*8}")

means = []
for seed in SEEDS:
    r = results[seed]
    if r["mean"] is not None:
        vl = r["val_losses"]
        mn, mx = min(vl), max(vl)
        spread = (mx - mn) / r["mean"] * 100
        print(f"  {seed:>6}  {r['mean']:>14.6f}  {mn:>12.6f}  {mx:>12.6f}  {spread:>7.2f}%")
        means.append(r["mean"])

if means:
    arr = np.array(means)
    print(f"\n  Across all seeds:")
    print(f"    Mean of means:  {arr.mean():.6f}")
    print(f"    Std of means:   {arr.std():.6f}")
    print(f"    CV (std/mean):  {arr.std()/arr.mean()*100:.2f}%")
    print(f"    Worst seed:     {SEEDS[np.argmax(arr)]} ({arr.max():.6f})")
    print(f"    Best seed:      {SEEDS[np.argmin(arr)]} ({arr.min():.6f})")
    print(f"    Range:          {(arr.max()-arr.min())/arr.mean()*100:.2f}%")
    print(f"""
  INTERPRETATION:
    CV < 1%   → very stable, sigmoid fix resolved seed sensitivity
    CV 1-5%   → mild sensitivity, acceptable for ensembling
    CV > 5%   → still seed-dependent, investigate further
{'='*80}""")


  SEED 73


  Seed 73: mean_val_loss=0.083365  (38.2 min)

  SEED TEST SUMMARY (base features, no enhancements)
    Seed   Mean Val Loss           Min           Max   Spread%
  ------  --------------  ------------  ------------  --------
      73        0.083365      0.083215      0.083441     0.27%

  Across all seeds:
    Mean of means:  0.083365
    Std of means:   0.000000
    CV (std/mean):  0.00%
    Worst seed:     73 (0.083365)
    Best seed:      73 (0.083365)
    Range:          0.00%

  INTERPRETATION:
    CV < 1%   → very stable, sigmoid fix resolved seed sensitivity
    CV 1-5%   → mild sensitivity, acceptable for ensembling
    CV > 5%   → still seed-dependent, investigate further


In [ ]:
#@title Save EVERYTHING from results/ as a single zip
import os, shutil

# Zip the entire results directory
zip_name = "mci_gru_all_results"
results_dir = "results"

if os.path.isdir(results_dir):
    shutil.make_archive(zip_name, "zip", ".", results_dir)
    size_mb = os.path.getsize(f"{zip_name}.zip") / 1e6

    # Show what's in there
    total_files = 0
    for root, dirs, files_list in os.walk(results_dir):
        total_files += len(files_list)
    print(f"Zipped: {zip_name}.zip  ({size_mb:.1f} MB, {total_files} files)")
    print(f"\nTop-level experiments:")
    for d in sorted(os.listdir(results_dir)):
        full = os.path.join(results_dir, d)
        if os.path.isdir(full):
            n = sum(len(f) for _, _, f in os.walk(full))
            print(f"  {d}/  ({n} files)")

    # Download
    try:
        from google.colab import files
        files.download(f"{zip_name}.zip")
        print(f"\n↓ Download started")
    except:
        print(f"\nZip saved locally as {zip_name}.zip")
else:
    print("No results/ directory found")

Zipped: mci_gru_all_results.zip  (10.3 MB, 2768 files)

Top-level experiments:
  seedtest_s73/  (2768 files)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


↓ Download started


In [ ]:
#@title Save ALL seed test data (predictions + logs + summary)
import os, glob, ast, json, shutil
import numpy as np
import pandas as pd

SEEDS = [42, 73, 7, 123, 256, 1, 999, 314, 55, 88]
rows = []
output_root = "results/seed_test_complete"
os.makedirs(output_root, exist_ok=True)

for seed in SEEDS:
    exp_root = f"results/seedtest_s{seed}"
    if not os.path.isdir(exp_root):
        continue
    runs = sorted(glob.glob(os.path.join(exp_root, "2*")))
    if not runs:
        continue
    run_dir = runs[-1]
    log_files = glob.glob(os.path.join(run_dir, "training_*.log"))
    if not log_files:
        continue

    log = open(log_files[0]).read()
    val_losses = []
    for line in log.splitlines():
        if "Best validation losses" in line:
            try:
                raw = "[" + line.split("[")[-1].split("]")[0] + "]"
                val_losses = ast.literal_eval(raw)
            except:
                pass

    pred_dir = os.path.join(run_dir, "averaged_predictions")
    n_preds = len(glob.glob(os.path.join(pred_dir, "*.csv"))) if os.path.isdir(pred_dir) else 0

    # Copy entire run directory into the bundle
    dest = os.path.join(output_root, f"seed_{seed}")
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest)

    rows.append({
        "seed": seed,
        "mean_val_loss": np.mean(val_losses) if val_losses else None,
        "std_val_loss": np.std(val_losses) if val_losses else None,
        "min_val_loss": min(val_losses) if val_losses else None,
        "max_val_loss": max(val_losses) if val_losses else None,
        "num_models": len(val_losses),
        "val_losses": val_losses,
        "n_prediction_days": n_preds,
    })

df = pd.DataFrame(rows)

if len(df) > 0 and df["mean_val_loss"].notna().any():
    means = df["mean_val_loss"].dropna()
    df_summary = pd.DataFrame([{
        "n_seeds": len(means),
        "grand_mean": means.mean(),
        "grand_std": means.std(),
        "cv_pct": means.std() / means.mean() * 100,
        "best_seed": df.loc[means.idxmin(), "seed"],
        "worst_seed": df.loc[means.idxmax(), "seed"],
        "range_pct": (means.max() - means.min()) / means.mean() * 100,
    }])
    df_summary.to_csv(os.path.join(output_root, "seed_summary.csv"), index=False)

df.to_csv(os.path.join(output_root, "seed_results.csv"), index=False)
df.to_pickle(os.path.join(output_root, "seed_results.pkl"))

config_info = {
    "features": "base (6 raw OHLCV+turnover, no momentum)",
    "architecture": "paper_faithful (no multi-scale, no self-attention, elu, latent_init=0.02)",
    "training": "num_models=3, num_epochs=15, early_stopping=5, batch_size=32, lr=5e-5",
    "data": "sp500_2019_universe_data.csv, train=2019-2023, val=2024, test=2025",
    "seeds": SEEDS,
}
with open(os.path.join(output_root, "config_info.json"), "w") as f:
    json.dump(config_info, f, indent=2)

# Show what's in the bundle
print("Bundle contents:")
for seed in SEEDS:
    seed_dir = os.path.join(output_root, f"seed_{seed}")
    if not os.path.isdir(seed_dir):
        print(f"  seed_{seed}: MISSING")
        continue
    pred_dir = os.path.join(seed_dir, "averaged_predictions")
    n_preds = len(glob.glob(os.path.join(pred_dir, "*.csv"))) if os.path.isdir(pred_dir) else 0
    n_ckpts = len(glob.glob(os.path.join(seed_dir, "checkpoints", "*.pth")))
    print(f"  seed_{seed}/  →  {n_preds} prediction CSVs, {n_ckpts} checkpoints, log + config")

print(f"\n{df.drop(columns=['val_losses']).to_string(index=False)}")
print(f"\n{df_summary.to_string(index=False)}")

# Zip and download
shutil.make_archive("seed_test_complete", "zip", output_root)
size_mb = os.path.getsize("seed_test_complete.zip") / 1e6
print(f"\nZip: seed_test_complete.zip ({size_mb:.1f} MB)")
try:
    from google.colab import files
    files.download("seed_test_complete.zip")
    print("↓ Download started")
except:
    print("Zip saved locally")

Bundle contents:
  seed_42: MISSING
  seed_73: MISSING
  seed_7/  →  252 prediction CSVs, 10 checkpoints, log + config
  seed_123/  →  252 prediction CSVs, 10 checkpoints, log + config
  seed_256: MISSING
  seed_1: MISSING
  seed_999: MISSING
  seed_314: MISSING
  seed_55: MISSING
  seed_88: MISSING

 seed  mean_val_loss  std_val_loss  min_val_loss  max_val_loss  num_models  n_prediction_days
    7       0.083478      0.000036      0.083415      0.083537          10                252
  123       0.083494      0.000060      0.083385      0.083579          10                252

 n_seeds  grand_mean  grand_std   cv_pct  best_seed  worst_seed  range_pct
       2    0.083486   0.000012 0.014015          7         123    0.01982

Zip: seed_test_complete.zip (20.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

↓ Download started


In [ ]:
#@title A · Upload seed_test_complete.zip and restore results
import os, shutil, zipfile
from google.colab import files

# Upload the zip
print("Upload seed_test_complete.zip:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract into a working directory
restore_dir = "seed_test_results"
if os.path.exists(restore_dir):
    shutil.rmtree(restore_dir)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(restore_dir)

# Show what we got
seeds_found = sorted([
    d for d in os.listdir(restore_dir)
    if d.startswith("seed_") and os.path.isdir(os.path.join(restore_dir, d))
])
print(f"\nRestored {len(seeds_found)} seed directories:")
for s in seeds_found:
    pred_dir = os.path.join(restore_dir, s, "averaged_predictions")
    n = len(os.listdir(pred_dir)) if os.path.isdir(pred_dir) else 0
    print(f"  {s}/averaged_predictions/  ({n} files)")

Upload seed_test_complete.zip:


Saving seed_test_complete.zip to seed_test_complete.zip

Restored 10 seed directories:
  seed_1/averaged_predictions/  (250 files)
  seed_123/averaged_predictions/  (250 files)
  seed_256/averaged_predictions/  (250 files)
  seed_314/averaged_predictions/  (250 files)
  seed_42/averaged_predictions/  (250 files)
  seed_55/averaged_predictions/  (250 files)
  seed_7/averaged_predictions/  (250 files)
  seed_73/averaged_predictions/  (250 files)
  seed_88/averaged_predictions/  (250 files)
  seed_999/averaged_predictions/  (250 files)


In [ ]:
#@title B · Backtest all 10 seeds + summary comparison
import subprocess, os, glob
import pandas as pd
import numpy as np

SEEDS = [42, 73, 7, 123, 256, 1, 999, 314, 55, 88]
restore_dir = "seed_test_results"

backtest_rows = []

for seed in SEEDS:
    seed_dir = os.path.join(restore_dir, f"seed_{seed}")
    pred_dir = os.path.join(seed_dir, "averaged_predictions")

    if not os.path.isdir(pred_dir):
        print(f"⚠ seed_{seed}: no averaged_predictions/, skipping")
        continue

    n_files = len(glob.glob(os.path.join(pred_dir, "*.csv")))
    print(f"\n{'='*80}")
    print(f"  BACKTEST: seed {seed}  ({n_files} prediction days)")
    print(f"{'='*80}\n")

    # Run backtest, capture output
    cmd = (
        f"python tests/backtest_sp500.py "
        f"--predictions_dir {pred_dir} "
        f"--data_file data/raw/market/sp500_2019_universe_data.csv "
        f"--test_start 2025-01-01 "
        f"--test_end 2025-12-31 "
        f"--top_k 20 "
        f"--label_t 5 "
        f"--auto_save "
        f"--backtest_suffix _seed{seed}"
    )
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    print(output[-2000:] if len(output) > 2000 else output)

    # Try to parse key metrics from output
    metrics = {"seed": seed}
    for line in output.splitlines():
        line_lower = line.lower()
        for metric in ["arr", "avol", "mdd", "asr", "cr", "ir"]:
            if metric in line_lower and ":" in line:
                try:
                    val = float(line.split(":")[-1].strip().replace("%", "").replace(",", ""))
                    metrics[metric.upper()] = val
                except:
                    pass
    backtest_rows.append(metrics)

# ── Summary table ──────────────────────────────────────────────────
df = pd.DataFrame(backtest_rows)
print(f"\n{'='*80}")
print(f"  BACKTEST SUMMARY ACROSS ALL SEEDS")
print(f"{'='*80}\n")

if len(df.columns) > 1:
    print(df.to_string(index=False))

    metric_cols = [c for c in df.columns if c != "seed"]
    if metric_cols:
        print(f"\n{'─'*60}")
        print(f"  {'Metric':>8}  {'Mean':>10}  {'Std':>10}  {'Min':>10}  {'Max':>10}  {'CV%':>8}")
        print(f"  {'─'*8}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*8}")
        for col in metric_cols:
            vals = df[col].dropna()
            if len(vals) > 0:
                cv = vals.std() / abs(vals.mean()) * 100 if vals.mean() != 0 else float("inf")
                print(f"  {col:>8}  {vals.mean():>10.4f}  {vals.std():>10.4f}  {vals.min():>10.4f}  {vals.max():>10.4f}  {cv:>7.1f}%")

        print(f"""
  INTERPRETATION:
    ASR CV < 20%  → strategy is robust across seeds
    ASR CV 20-50% → moderate seed sensitivity, ensemble helps
    ASR CV > 50%  → high sensitivity, architecture needs work
{'='*80}""")
else:
    print("Could not parse metrics from backtest output.")
    print("Check the raw output above for results.")

# Save summary
os.makedirs("backtest_seed_summary", exist_ok=True)
df.to_csv("backtest_seed_summary/backtest_all_seeds.csv", index=False)
print(f"\nSaved to backtest_seed_summary/backtest_all_seeds.csv")

try:
    from google.colab import files
    files.download("backtest_seed_summary/backtest_all_seeds.csv")
except:
    pass


  BACKTEST: seed 42  (250 prediction days)

eed_42/backtest_seed42/monthly_performance.csv
  Daily holdings: seed_test_results/seed_42/backtest_seed42/daily_holdings.csv
  Return attribution: seed_test_results/seed_42/backtest_seed42/return_attribution.csv
  Portfolio composition: seed_test_results/seed_42/backtest_seed42/portfolio_composition.csv
  Holdings summary: seed_test_results/seed_42/backtest_seed42/holdings_summary.csv
  Trade journal: seed_test_results/seed_42/backtest_seed42/trade_journal.csv
  Summary: seed_test_results/seed_42/backtest_seed42/summary.txt

✓ All backtest outputs saved successfully!
Loading predictions from seed_test_results/seed_42/averaged_predictions...
  Loaded 118000 predictions
  250 dates, 472 stocks
  Date range: 2025-01-02 to 2025-12-31
Calculating 5-day forward returns...
  Added returns: forward_return_5d, open_to_open_return, daily_return, overnight_gap
  Portfolio & Benchmark use 'open_to_open_return' (same entry/exit window)
  'daily_return' 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>